# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()

print(f"Dataset: {getattr(dataset.metadata, 'name', '')}\nDescription: {getattr(dataset.metadata, 'description', '')}\n\nPublished Date: {getattr(dataset.metadata, 'datePublished', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Each data table (RecordSet) and field is identified by a unique `@id` in the Croissant schema. Let's explore what's available.

In [ ]:
# List all available record sets and their fields
record_sets = list(dataset.record_sets)
print("Record Sets (@id and name):\n==========================")
for record_set in record_sets:
    print(f"- {record_set.id} | {record_set.name}")
    print("  Fields/Columns:")
    for field in record_set.fields:
        print(f"    - {field.id} | {field.name} | type: {field.data_type}")
    print()
# For further processing, select the first RecordSet by its id.
example_record_set_id = record_sets[0].id if record_sets else None

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

> ⚡ **Note:** All references to record sets and fields are made using their `@id` values.


In [ ]:
# Extract data from all record sets
all_record_set_ids = [rs.id for rs in record_sets]
dataframes = {}
for rsid in all_record_set_ids:
    records = list(dataset.records(record_set=rsid))
    if records:
        dataframes[rsid] = pd.DataFrame(records)
    else:
        print(f"No records extracted for RecordSet: {rsid}")
    
# Display columns for the first record set (if any records)
if example_record_set_id and example_record_set_id in dataframes:
    print(f"First 5 columns for {example_record_set_id}:")
    print(dataframes[example_record_set_id].columns[:5].tolist())
    dataframes[example_record_set_id].head()
else:
    print("No suitable record set found for data extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping records.

Let's select a numeric field and a group field, referencing them by their `@id`. We'll show their IDs in the previous overview cell output.

In [ ]:
# For demonstration, we select the first numeric field and the first categorical field (if available)
import numpy as np

if example_record_set_id and example_record_set_id in dataframes:
    record_set = None
    for rs in record_sets:
        if rs.id == example_record_set_id:
            record_set = rs
            break
    numeric_field = None
    group_field = None
    for field in record_set.fields:
        # Prefer float > int for numeric
        if not numeric_field and (field.data_type.lower() in ['number','float','integer']):
            if field.id in dataframes[example_record_set_id].columns:
                numeric_field = field.id
        # Use string/categorical for grouping
        if not group_field and (field.data_type.lower() in ['string','text']) and field.id in dataframes[example_record_set_id].columns:
            group_field = field.id
    
    print(f"Numeric field candidate for EDA: {numeric_field}")
    print(f"Grouping field candidate for EDA: {group_field}")

    # Proceed if both found
    if numeric_field:
        df = dataframes[example_record_set_id]
        # Attempt numeric conversion
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].quantile(0.75)
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (75th percentile):\n")
        print(filtered_df.head())
        
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, norm_col]].head())
        
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean {numeric_field} by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example using `matplotlib` and `seaborn` to plot the distribution of the selected numeric field, grouped by a categorical variable if present.


In [ ]:
# Visualize distribution and group differences for the selected numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if example_record_set_id and example_record_set_id in dataframes and numeric_field:
    df_viz = dataframes[example_record_set_id].copy()
    df_viz[numeric_field] = pd.to_numeric(df_viz[numeric_field], errors='coerce')
    df_viz = df_viz.dropna(subset=[numeric_field])
    plt.figure(figsize=(8, 4))
    sns.histplot(df_viz[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    # If a grouping/categorical field is present, show boxplot
    if group_field:
        plt.figure(figsize=(10,5))
        # Limit to top 10 groups for readability
        top_groups = df_viz[group_field].value_counts().nlargest(10).index
        df_group = df_viz[df_viz[group_field].isin(top_groups)]
        sns.boxplot(data=df_group, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field} (top 10 groups)")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Insufficient data to visualize.")

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and analyze the FAIR² protein mass spectrometry dataset using the `mlcroissant` library. 

We referenced all schema entities by their `@id`s consistently, as recommended for reproducible data science. Key fields and record sets were identified and explored, allowing extraction and basic analysis of this dataset for downstream applications, such as protein abundance profiling and group comparisons.